# Replication notebook
This notebook walks through the package step by step: data loading, feature construction, sample splitting, model estimation, and evaluation.

In [ ]:
from __future__ import annotations
from pathlib import Path

import logging
from pandas.api.types import is_numeric_dtype
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pandas import Timestamp
from typing import Any

from config import (
    CacheConfig,
    TimeframeConfig,
    DataFilesConfig,
    RunControlConfig,
    SplitConfig,
    DataRegimeConfig,
    HyperGridConfig,
    ReproducibilityConfig,
    CharacteristicsFrequency,
    LoggingConfig,
    ExpandingWindowConfig,
    ModelSelectionConfig,
)

from io_utils import ensure_dir, save_parquet, set_global_seed, setup_project_logger
from data_inputs import load_datashare, load_crsp_monthly, load_macro_monthly
from dataset_builder import impute_characteristics_by_month_cross_sectional_median, compute_missingness_for_characteristics, save_missingness_comparison_plot, summarize_columns, save_missingness_three_comparison_plot, summary_stats_extended


In [3]:
repro_cfg = ReproducibilityConfig(random_state=42)
cache_cfg = CacheConfig()
tf_cfg = TimeframeConfig()
data_cfg = DataFilesConfig()
run_ctrl_cfg = RunControlConfig()
split_cfg = SplitConfig()
regime_cfg = DataRegimeConfig(mode="full")
grid_cfg = HyperGridConfig(use_extended_grids=False)
freq_cfg = CharacteristicsFrequency()
log_cfg = LoggingConfig()
expand_cfg = ExpandingWindowConfig()
model_cfg = ModelSelectionConfig()


In [4]:

# Set global seed ONCE at the start - ensures reproducibility across all runs
set_global_seed(repro_cfg.random_state, repro_cfg.torch_deterministic)

logger, log_path = setup_project_logger(
    logger_name=log_cfg.logger_name,
    log_dir=log_cfg.log_dir,
    regime_mode=regime_cfg.mode,
    overwrite_log=log_cfg.overwrite_log,
    file_level_full=log_cfg.file_level_full,
    file_level_coding=log_cfg.file_level_coding,
    console_level_full=log_cfg.console_level_full,
    console_level_coding=log_cfg.console_level_coding,
)

logger.info("Starting experiment run.")
logger.info(f"Random seed: {repro_cfg.random_state}")
logger.info(f"Cache dir: {cache_cfg.cache_dir}")
logger.info(f"Data regime: {regime_cfg.mode}")


2026-09-18 20:15:50 | INFO | eap_ml | Logger initialized.
2026-09-18 20:15:50 | INFO | eap_ml | Regime mode: full
2026-09-18 20:15:50 | INFO | eap_ml | Log file: logs\eap_ml_full_20260918_201550.log
2026-09-18 20:15:50 | INFO | eap_ml | Starting experiment run.
2026-09-18 20:15:50 | INFO | eap_ml | Random seed: 42
2026-09-18 20:15:50 | INFO | eap_ml | Cache dir: cache
2026-09-18 20:15:50 | INFO | eap_ml | Data regime: full


In [5]:
ensure_dir("output")
ensure_dir(cache_cfg.cache_dir)

In [6]:
complete_dataset_path=f"{cache_cfg.cache_dir}/complete_dataset.parquet"
descriptives_path=cache_cfg.descriptives_dir

feature_panel_path = f"{cache_cfg.cache_dir}/feature_panel_{regime_cfg.mode}.parquet"
feature_cols_path = f"{cache_cfg.cache_dir}/feature_cols_{regime_cfg.mode}.pkl"

In [7]:
datashare_path=data_cfg.datashare_path
crsp_path=data_cfg.crsp_monthly_path
macro_path=data_cfg.macro_path
out_path=complete_dataset_path
descriptives_path=descriptives_path
cache_enabled=cache_cfg.enabled
possible_crsp_cols=data_cfg.possible_crsp_cols
possible_marco_cols=data_cfg.possible_marco_cols
cols_chara=data_cfg.chara_cols
cols_vars_monthly=freq_cfg.cols_vars_monthly
cols_vars_quarterly=freq_cfg.cols_vars_quarterly
cols_vars_annual=freq_cfg.cols_vars_annual

In [8]:
logger = logging.getLogger("eap_ml.dataset_builder")

In [9]:
logger.info(f"Building complete dataset, output: {out_path}")


logger.debug("Loading source datasets")

crsp = load_crsp_monthly(crsp_path, possible_crsp_cols)
macro = load_macro_monthly(macro_path, possible_marco_cols)
ds = load_datashare(datashare_path)


2026-09-18 20:15:51 | INFO | eap_ml.dataset_builder | Building complete dataset, output: cache/complete_dataset.parquet
2026-09-18 20:15:51 | DEBUG | eap_ml.dataset_builder | Loading source datasets
2026-09-18 20:15:51 | DEBUG | eap_ml.data_inputs | Loading CRSP monthly data from C:/Coding/Project/data/crsp_monthly.csv
c:\Coding\thesis-project\data_inputs.py:120: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
2026-09-18 20:16:01 | DEBUG | eap_ml.data_inputs | Standardizing date of CRSP
2026-09-18 20:16:15 | DEBUG | eap_ml.data_inputs | CRSP loaded: 4594389 rows
2026-09-18 20:16:15 | DEBUG | eap_ml.data_inputs | Loading macro data from C:/Coding/Project/data/Data2024_monthly_goyal.csv
2026-09-18 20:16:15 | DEBUG | eap_ml.data_inputs | Standardizing date of Macro
2026-09-18 20:16:15 | DEBUG | eap_ml.data_inputs | Macro loaded: 1848 rows, columns: ['date', 'tbl', 'd/p', 'e/p', 'b/m', 'tms', 'dfy', 'ntis', 'svar'

In [10]:

#-------------
date_1987_05 = pd.Period("1987-05", freq="M")
#-------------
#-------------
test1 = ds.loc[ds["date"] == date_1987_05].copy()
#-------------


In [11]:

# Columns 3-96 in datashare.csv = 94 characteristics.
characteristic_cols = cols_chara
logger.debug(f"Identified {len(characteristic_cols)} characteristic columns")

logger.debug("Merging datashare with CRSP")
merged = ds.merge(
    crsp[["permno", "date", "ret", "dlret"]],
    on=["permno", "date"],
    how="inner",
    # validate="one_to_one",
)


2026-09-18 20:18:05 | DEBUG | eap_ml.dataset_builder | Identified 94 characteristic columns
2026-09-18 20:18:05 | DEBUG | eap_ml.dataset_builder | Merging datashare with CRSP


In [12]:

#-------------
test2 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------

logger.debug("Merging with macro data")
merged = merged.merge(macro, on="date", how="left")

merged = merged.sort_values(["permno", "date"]).reset_index(drop=True)



2026-09-18 20:18:23 | DEBUG | eap_ml.dataset_builder | Merging with macro data


In [13]:

#-------------
test3 = merged.loc[merged["date"] == date_1987_05].copy()
#-------------


In [14]:

# ------------------------------------------------------------------
# ret_total
# ------------------------------------------------------------------
logger.debug("Creating ret_total")

has_return_data = (merged["ret"].notna() | merged["dlret"].notna())

merged["ret_total"] = (
    (1.0 + merged["ret"].fillna(0.0))
    * (1.0 + merged["dlret"].fillna(0.0))
    - 1.0
).where(has_return_data)    


2026-09-18 20:20:07 | DEBUG | eap_ml.dataset_builder | Creating ret_total


In [15]:
#-------------
tracked_cols = ["ret_total", *cols_vars_monthly, *cols_vars_quarterly, *cols_vars_annual]
#-------------
test4 = merged.loc[merged["date"] == date_1987_05].copy()
missing_4 = merged[tracked_cols].isna().copy()
#-------------


In [ ]:
summary_1 = summ

In [16]:

# ------------------------------------------------------------------
# Missingness visualisation and imputation
# ------------------------------------------------------------------
cols_chara_and_ret_total = characteristic_cols + ["ret_total"]

# Visualisation before imputation
missing_before = compute_missingness_for_characteristics(merged, cols_chara_and_ret_total)

before_csv = f"{descriptives_path}/charas_missingness_before.csv"
missing_before.to_csv(before_csv, index=False)

#-------------
test5 = merged.loc[merged["date"] == date_1987_05].copy()
missing_5 = merged[tracked_cols].isna().copy()
#-------------


In [17]:


logger.debug(f"Saved missingness before imputation: {before_csv}")

# Imputation
logger.info("Imputing missing characteristics using monthly cross-sectional medians")
merged = impute_characteristics_by_month_cross_sectional_median(merged, cols_chara_and_ret_total)

#-------------
test6 = merged.loc[merged["date"] == date_1987_05].copy()
missing_6 = merged[tracked_cols].isna().copy()
#-------------

# Visualisation after imputation
missing_after = compute_missingness_for_characteristics(merged, cols_chara_and_ret_total)

after_csv = f"{descriptives_path}/charas_missingness_after.csv"
missing_after.to_csv(after_csv, index=False)


logger.debug(f"Saved missingness after imputation: {after_csv}")

# Comparison plot
comparison_jpg = f"{descriptives_path}/charas_missingness_comparison.jpg"
save_missingness_comparison_plot(
    missing_before,
    missing_after,
    comparison_jpg,
    title="Missing Data Percentage: Before vs After Imputation",
)
logger.debug(f"Saved missingness comparison plot: {comparison_jpg}")



2026-09-18 20:20:23 | DEBUG | eap_ml.dataset_builder | Saved missingness before imputation: descriptives/charas_missingness_before.csv
2026-09-18 20:20:23 | INFO | eap_ml.dataset_builder | Imputing missing characteristics using monthly cross-sectional medians
2026-09-18 20:21:28 | DEBUG | eap_ml.dataset_builder | Saved missingness after imputation: descriptives/charas_missingness_after.csv
2026-09-18 20:21:29 | DEBUG | eap_ml.dataset_builder | Saved missingness comparison plot: descriptives/charas_missingness_comparison.jpg


In [18]:

# ------------------------------------------------------------------
# Temporal shifts 
# ------------------------------------------------------------------


#-------------
test7 = merged.loc[merged["date"] == date_1987_05].copy()
missing_7 = merged[tracked_cols].isna().copy()
#-------------


In [19]:

# Build next-month excess return target.
logger.debug("Building lead return target (shift ret_total)")


# Build shifts for monthly, quarterly and annual characteristcs
logger.debug("Building characteristic shifts and ret_total shift")
merged = merged.sort_values(["permno", "date"]).copy()
grouped = merged.groupby("permno")

for i in merged.columns:
    if i in cols_vars_monthly or i == "ret_total":
        merged[i] = merged.groupby("permno")[i].shift(-1)
    elif i in cols_vars_quarterly:
        merged[i] = merged.groupby("permno")[i].shift(-3)
    elif i in cols_vars_annual:
        merged[i] = merged.groupby("permno")[i].shift(-6)


#-------------
test8 = merged.loc[merged["date"] == date_1987_05].copy()
missing_8 = merged[tracked_cols].isna().copy()
#-------------


2026-09-18 20:21:34 | DEBUG | eap_ml.dataset_builder | Building lead return target (shift ret_total)
2026-09-18 20:21:34 | DEBUG | eap_ml.dataset_builder | Building characteristic shifts and ret_total shift


In [20]:

# ------------------------------------------------------------------
# Summary 
# ------------------------------------------------------------------
summary_before = summarize_columns(merged, cols_chara_and_ret_total)

summary_before_csv = f"{descriptives_path}/summary_before.csv"
summary_before.to_csv(summary_before_csv, index=False)

# Define inclusive monthly boundaries
start_period = pd.Period("1957-10", freq="M")
end_period = pd.Period("2021-06", freq="M")

# Keep observations from October 1957 through June 2021

mask = (
    (merged["date"] >= start_period)
    & (merged["date"] <= end_period)
)
merged = merged.loc[mask].copy()

logger.debug(f"Dataset reduced due to missing values to : {merged["date"].min()} and {merged["date"].max()}")



2026-09-18 20:24:24 | DEBUG | eap_ml.dataset_builder | Dataset reduced due to missing values to : 1957-10 and 2021-06


In [21]:

#-------------
test9 = merged.loc[merged["date"] == date_1987_05].copy()
missing_9 = merged[tracked_cols].isna().copy()
#-------------


In [22]:

# Visualisation of missingness after temporal cuts
missing_after_cut = compute_missingness_for_characteristics(merged, cols_chara_and_ret_total)

after_cut_csv = f"{descriptives_path}/charas_missingness_after_cut.csv"
missing_after_cut.to_csv(after_cut_csv, index=False)

logger.debug(f"Saved missingness after temporal cut: {after_cut_csv}")

# Comparison plot
comparison_three_jpg = f"{descriptives_path}/charas_missingness_comparison_three.jpg"
save_missingness_three_comparison_plot(
    missing_before,
    missing_after,
    missing_after_cut,
    comparison_three_jpg,
    title="Missing Data Percentage: Before vs After Imputation vs After Temporal Reduction",
)
logger.debug(f"Saved missingness comparison plot: {comparison_jpg}")



2026-09-18 20:24:38 | DEBUG | eap_ml.dataset_builder | Saved missingness after temporal cut: descriptives/charas_missingness_after_cut.csv
2026-09-18 20:24:39 | DEBUG | eap_ml.dataset_builder | Saved missingness comparison plot: descriptives/charas_missingness_comparison.jpg


In [23]:

#-------------
test10 = merged.loc[merged["date"] == date_1987_05].copy()
missing_10 = merged[tracked_cols].isna().copy()
#-------------


In [24]:


test1_csv = f"{descriptives_path}/test1.csv"
test1.to_csv(test1_csv, index=False)

test2_csv = f"{descriptives_path}/test2.csv"
test2.to_csv(test2_csv, index=False)

test3_csv = f"{descriptives_path}/test3.csv"
test3.to_csv(test3_csv, index=False)

test4_csv = f"{descriptives_path}/test4.csv"
test4.to_csv(test4_csv, index=False)

test5_csv = f"{descriptives_path}/test5.csv"
test5.to_csv(test5_csv, index=False)

test6_csv = f"{descriptives_path}/test6.csv"
test6.to_csv(test6_csv, index=False)

test7_csv = f"{descriptives_path}/test7.csv"
test7.to_csv(test7_csv, index=False)

test8_csv = f"{descriptives_path}/test8.csv"
test8.to_csv(test8_csv, index=False)

test9_csv = f"{descriptives_path}/test9.csv"
test9.to_csv(test9_csv, index=False)

test10_csv = f"{descriptives_path}/test9.csv"
test10.to_csv(test9_csv, index=False)


In [25]:
#-------------
newly_missing_5 = (
    ~missing_4
    & missing_5
)

newly_missing_rows_5 = merged.loc[
    newly_missing_5.any(axis=1),
    ["permno", "date", *tracked_cols],
]
print("newly_missing_rows_5")
print(newly_missing_rows_5.head(20))

# i = 6
newly_missing_6 = (
    ~missing_5
    & missing_6
)

newly_missing_rows_6 = merged.loc[
    newly_missing_6.any(axis=1),
    ["permno", "date", *tracked_cols],
]
print("newly_missing_rows_6")
print(newly_missing_rows_6.head(20))

# i = 7
newly_missing_7 = (
    ~missing_6
    & missing_7
)

newly_missing_rows_7 = merged.loc[
    newly_missing_7.any(axis=1),
    ["permno", "date", *tracked_cols],
]
print("newly_missing_rows_7")
print(newly_missing_rows_7.head(20))

# i = 8
newly_missing_8 = (
    ~missing_7
    & missing_8
)

newly_missing_rows_8 = merged.loc[
    newly_missing_8.any(axis=1),
    ["permno", "date", *tracked_cols],
]
print("newly_missing_rows_8")
print(newly_missing_rows_8.head(20))

# i = 9
newly_missing_9 = (
    ~missing_8
    & missing_9
)

newly_missing_rows_9 = merged.loc[
    newly_missing_9.any(axis=1),
    ["permno", "date", *tracked_cols],
]
print("newly_missing_rows_9")
print(newly_missing_rows_9.head(20))

# i = 10
newly_missing_10 = (
    ~missing_9
    & missing_10
)

newly_missing_rows_10 = merged.loc[
    newly_missing_10.any(axis=1),
    ["permno", "date", *tracked_cols],
]
print("newly_missing_rows_10")
print(newly_missing_rows_10.head(20))


newly_missing_rows_5
Empty DataFrame
Columns: [permno, date, ret_total, baspread, beta, betasq, chmom, dolvol, idiovol, ill, indmom, maxret, mom12m, mom1m, mom36m, mom6m, mvel1, pricedelay, retvol, std_dolvol, std_turn, turn, zerotrade, aeavol, cash, chtx, cinvest, ear, ms, nincr, roaq, roavol, roeq, rsup, stdacc, stdcf, absacc, acc, age, agr, bm, bm_ia, cashdebt, cashpr, cfp, cfp_ia, chatoia, chcsho, chempia, chinv, chpmia, convind, currat, depr, divi, divo, dy, egr, ep, gma, grcapx, grltnoa, herf, hire, invest, lev, lgr, mve_ia, operprof, orgcap, pchcapx_ia, pchcurrat, pchdepr, pchgm_pchsale, pchquick, pchsale_pchinvt, pchsale_pchrect, pchsale_pchxsga, pchsaleinv, pctacc, ps, quick, rd, rd_mve, rd_sale, realestate, roic, salecash, saleinv, salerec, secured, securedind, sgr, sin, sp, tang, tb]
Index: []

[0 rows x 97 columns]
newly_missing_rows_6
Empty DataFrame
Columns: [permno, date, ret_total, baspread, beta, betasq, chmom, dolvol, idiovol, ill, indmom, maxret, mom12m, mom1m, mom36

In [ ]:


logger.info(f"Complete dataset built: {merged.shape}")
save_parquet(merged, out_path, enabled=cache_enabled)

In [ ]:
merged["date"] = pd.to_datetime(merged["date"])



for i in merged.columns:
    if i in cols_vars_monthly:
        new_column = (merged.set_index("date")
            .groupby("permno")[i]
            .shift(-1, freq=pd.DateOffset(months=1)))
        merged.set_index(["permno", "date"], inplace=True)
        merged[i] = new_column
        merged.reset_index(inplace=True)
    elif i in cols_vars_quarterly:
        new_column = (merged.set_index("date")
            .groupby("permno")[i]
            .shift(-1, freq=pd.DateOffset(months=1)))
        merged.set_index(["permno", "date"], inplace=True)
        merged[i] = new_column
        merged.reset_index(inplace=True)
    elif i in cols_vars_annual:
        new_column = (merged.set_index("date")
            .groupby("permno")[i]
            .shift(-1, freq=pd.DateOffset(months=1)))
        merged.set_index(["permno", "date"], inplace=True)
        merged[i] = new_column
        merged.reset_index(inplace=True)


"""
new_column = (merged.set_index("date")
                .groupby("permno")[i]
                .shift(-1, freq=pd.DateOffset(months=1)))
merged.set_index(["permno", "date"], inplace=True)
merged[i] = new_column
merged.reset_index(inplace=True)
"""

merged["date"] = merged["date"].dt.to_period("M")

In [ ]:
tracked_cols = ["ret_total", *cols_vars_monthly, *cols_vars_quarterly, *cols_vars_annual]
missing_before = merged[tracked_cols].isna().copy()



missing_after = merged[tracked_cols].isna()
newly_missing = (
    ~missing_before
    & missing_after
)



newly_missing_rows = merged.loc[
    newly_missing.any(axis=1),
    ["permno", "date", *tracked_cols],
]
print(newly_missing_rows.head(20))

In [ ]:
merged["date"] = pd.to_datetime(merged["date"])
new_column = (merged.set_index("date")
                .groupby("permno")[i]
                .shift(-1, freq=pd.DateOffset(months=1)))
merged.set_index(["permno", "date"], inplace=True)
merged[i] = new_column
merged.reset_index(inplace=True)
merged["date"] = merged["date"].dt.to_period("M")